In [1]:
import scipy.io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# 1. Load the raw MATLAB file
mat_data = scipy.io.loadmat('../data/raw/GEIS.mat', squeeze_me=True, struct_as_record=False)

# 2. Initialize a list to collect the records
records = []

# Mapping of file keys to the actual temperature values
temperatures = {
    'GEIS_20': 20.0, 'GEIS_22_5': 22.5, 'GEIS_25': 25.0, 
    'GEIS_27_5': 27.5, 'GEIS_30': 30.0, 'GEIS_35': 35.0, 
    'GEIS_40': 40.0, 'GEIS_47_5': 47.5
}

# 3. Extract data from the MATLAB structure
for aging_idx in range(5):
    aging_key = f'Aging{aging_idx}'
    aging_struct = getattr(mat_data, aging_key, mat_data[aging_key])
    
    for temp_key, temp_val in temperatures.items():
        data = getattr(aging_struct, temp_key)
        for soc_idx, d in enumerate(data):
            for freq_idx, row in enumerate(d):
                freq = row[0]
                z_real = row[1] * 1000 # Convert to mOhm as in the paper
                neg_z_imag = row[2] * 1000 # Convert to mOhm (raw .mat stores already -Im(Z))
                
                records.append({
                    'Aging': aging_idx,
                    'Temperature': temp_val,
                    'SOC': soc_idx,  # 0 to 4 (approximately 0%, 25%, 50%, 75%, 100%)
                    'Frequency': freq,
                    'Z_real': z_real,
                    'neg_Z_imag': neg_z_imag
                })

# 4. Build the cleaned DataFrame
df = pd.DataFrame(records)

# Save as CSV for future use / backup
df.to_csv('../data/processed/batteries_cleaned_dataset.csv', index=False)

# Display the first rows of the cleaned dataset
display(df.head())

,Aging,Temperature,SOC,Frequency,Z_real,neg_Z_imag
0,0,20.0,0,10002.2290,3.004527,-0.916100
1,0,20.0,0,7904.9590,2.971508,-0.661783
2,0,20.0,0,6248.7827,3.047303,-0.422141
3,0,20.0,0,4941.5811,3.100461,-0.164312
4,0,20.0,0,3906.2498,3.172908,-0.012017
